# 1-Bosqich: Chang bo'roni hodisalarini aniqlash (yangilangan)

Bu notebook 0-bosqichdagi muammolarni hal qiladi:
1. **Tog' stansiyalarini chiqarib tashlash** — ular qor bo'roni/tuman ko'rsatadi
2. **Terrain type bo'yicha alohida mezonlar** — cho'l vs tekislik
3. **Harorat filtri** — sovuq kunlar = qor/tuman, issiq kunlar = chang
4. **Stansiyalar xaritasi** — hodisalar geografiyasi
5. **Yakuniy "label" jadvali** — ML model uchun tayyor

---

## 1. Kutubxonalar va sozlamalar

In [ ]:
!pip install pandas xlrd openpyxl matplotlib seaborn folium --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print("Kutubxonalar yuklandi!")

In [ ]:
# ============================================================
# YO'LLAR — o'z kompyuteringizga moslang!
# ============================================================
DATA_DIR = "../data/все станции за 10 лет/"
COORDS_FILE = "../data/stations_coordinates.csv"

# Tekshirish
assert os.path.exists(DATA_DIR), f"DATA_DIR topilmadi: {DATA_DIR}"
assert os.path.exists(COORDS_FILE), f"COORDS_FILE topilmadi: {COORDS_FILE}"
print("✅ Barcha yo'llar to'g'ri!")

## 2. Stansiya koordinatalarini yuklash va terrain bo'yicha ajratish

In [ ]:
# Koordinatalar jadvali
coords = pd.read_csv(COORDS_FILE)

print(f"📍 Jami stansiyalar: {len(coords)}")
print(f"\nTerrain type bo'yicha taqsimot:")
print(coords['terrain_type'].value_counts().to_string())

print(f"\n\nTOG' stansiyalari (CHIQARIB TASHLANADI):")
tog = coords[coords['terrain_type'] == 'tog']
print(tog[['station', 'elevation_m', 'viloyat']].to_string(index=False))

print(f"\n\nCHO'L stansiyalari (CHANG BO'RONI MANBAI):")
chol = coords[coords['terrain_type'] == 'chol']
print(chol[['station', 'elevation_m', 'viloyat']].to_string(index=False))

In [ ]:
# Tog' stansiyalarini chiqarib tashlash
tog_stations = coords[coords['terrain_type'] == 'tog']['station'].tolist()

# Qolgan stansiyalar (cho'l + tekislik + tog_oldi)
valid_stations = coords[coords['terrain_type'] != 'tog']['station'].tolist()

print(f"\n🚫 Chiqarilgan (tog'): {len(tog_stations)} stansiya")
print(f"✅ Qolgan: {len(valid_stations)} stansiya")
print(f"   - Cho'l: {len(coords[coords['terrain_type'] == 'chol'])}")
print(f"   - Tekislik: {len(coords[coords['terrain_type'] == 'tekislik'])}")
print(f"   - Tog' oldi: {len(coords[coords['terrain_type'] == 'tog_oldi'])}")

## 3. Ma'lumotlarni yuklash (faqat valid stansiyalar)

In [ ]:
def load_station(filepath):
    """Bitta stansiya faylini o'qish."""
    fname = os.path.basename(filepath)
    try:
        engine = 'openpyxl' if fname.endswith('.xlsx') else 'xlrd'
        df_test = pd.read_excel(filepath, header=None, nrows=5, engine=engine)
        
        header_row = None
        for idx in range(len(df_test)):
            if 'Year' in df_test.iloc[idx].astype(str).str.strip().tolist():
                header_row = idx
                break
        
        if header_row is None:
            header_row = 0
        
        df = pd.read_excel(filepath, header=header_row, engine=engine)
        df.columns = [str(c).strip() for c in df.columns]
        df['station'] = os.path.splitext(fname)[0].strip()
        return df
    except Exception as e:
        print(f"  ⚠️ {fname}: {e}")
        return None

In [ ]:
# Faqat valid stansiyalarni yuklash
all_data = []
files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith(('.xls', '.XLS', '.xlsx'))])

print(f"Yuklanmoqda (tog'siz): ...\n")

loaded = 0
skipped_tog = 0

for fname in files:
    station_name = os.path.splitext(fname)[0].strip()
    
    # Tog' stansiyasimi?
    if station_name in tog_stations:
        skipped_tog += 1
        continue
    
    filepath = os.path.join(DATA_DIR, fname)
    df = load_station(filepath)
    
    if df is not None:
        all_data.append(df)
        loaded += 1

print(f"\n✅ Yuklangan: {loaded} stansiya")
print(f"🚫 Tog' (o'tkazilgan): {skipped_tog} stansiya")

In [ ]:
# Birlashtirish
df_all = pd.concat(all_data, ignore_index=True)

# Sana ustuni
df_all['Year'] = pd.to_numeric(df_all['Year'], errors='coerce').astype('Int64')
df_all['Mon'] = pd.to_numeric(df_all['Mon'], errors='coerce').astype('Int64')
df_all['Day'] = pd.to_numeric(df_all['Day'], errors='coerce').astype('Int64')
df_all['date'] = pd.to_datetime(
    df_all[['Year', 'Mon', 'Day']].rename(columns={'Year':'year','Mon':'month','Day':'day'}),
    errors='coerce'
)

# Raqamli ustunlar
num_cols = ['V', 'Vx8', 'VxG', 'U', 'UN', 'Ed', 'Taav', 'Tg', 'StP', 'SeP', 'TaN', 'TaX']
for col in num_cols:
    if col in df_all.columns:
        df_all[col] = pd.to_numeric(df_all[col], errors='coerce')

# Terrain type qo'shish
df_all = df_all.merge(coords[['station', 'terrain_type', 'elevation_m', 'lat', 'lon', 'viloyat']], 
                       on='station', how='left')

# Faqat 2011-2020
df_all = df_all[(df_all['Year'] >= 2011) & (df_all['Year'] <= 2020)].copy()

print(f"📊 MA'LUMOT:")
print(f"   Qatorlar: {len(df_all):,}")
print(f"   Stansiyalar: {df_all['station'].nunique()}")
print(f"   Davr: {df_all['date'].min().date()} — {df_all['date'].max().date()}")
print(f"   Terrain: {df_all['terrain_type'].value_counts().to_dict()}")

## 4. Yangilangan chang bo'roni aniqlash algoritmi

### Mezonlar:
1. **Tuman filtri:** V past + shamol < 5 m/s + namlik > 70% → TUMAN
2. **Harorat filtri:** Taav < 0°C → qor bo'roni ehtimoli (chang emas)
3. **Cho'l uchun:** mezonlar yengilroq (cho'lda chang tez-tez)
4. **Tekislik uchun:** mezonlar standart
5. **Tog' oldi:** mezonlar qattiqroq

In [ ]:
def detect_dust_storms_v2(df):
    """
    Yangilangan chang bo'roni aniqlash algoritmi.
    
    Yangiliklar (0-bosqichga nisbatan):
    - Harorat filtri: Taav < 0°C = qor bo'roni, chang emas
    - Terrain-adaptive mezonlar
    - Yaxshilangan tuman filtri
    """
    df = df.copy()
    
    # Yordamchi ustunlar
    df['wind'] = df['VxG'].fillna(df['Vx8'])
    df['humidity'] = df['UN'].fillna(df['U'])
    
    # ═══════════════════════════════════════════════════
    # FILTRLAR (false positive ni chiqarish)
    # ═══════════════════════════════════════════════════
    
    # 1. TUMAN FILTRI: past ko'rinish + past shamol + yuqori namlik
    is_fog = (
        (df['V'] < 2) & 
        (df['wind'].fillna(0) < 5) & 
        (df['humidity'].fillna(100) > 70)
    )
    
    # 2. QOR BO'RONI FILTRI: harorat 0°C dan past
    is_snow = (
        (df['V'] < 2) & 
        (df['Taav'].fillna(10) < 0)
    )
    
    # 3. Ikkala filtrni birlashtirish
    exclude_mask = is_fog | is_snow
    
    # ═══════════════════════════════════════════════════
    # TERRAIN-ADAPTIVE MEZONLAR
    # ═══════════════════════════════════════════════════
    
    # Cho'l stansiyalari uchun (quruqlik yuqori, chang tez-tez)
    is_chol = df['terrain_type'] == 'chol'
    
    # Cho'l uchun: faqat V + shamol yetarli (namlik doim past)
    chol_kuchli = is_chol & (~exclude_mask) & (df['V'] < 0.5) & (df['wind'] >= 8)
    chol_ortacha = is_chol & (~exclude_mask) & (~chol_kuchli) & (df['V'] < 1.0) & (df['wind'] >= 6)
    chol_yengil = is_chol & (~exclude_mask) & (~chol_kuchli) & (~chol_ortacha) & (
        (df['V'] < 2.0) & (df['wind'] >= 8) & (df['humidity'].fillna(50) < 50)
    )
    
    # Tekislik va tog' oldi uchun: standart mezonlar
    is_other = ~is_chol
    
    other_kuchli = is_other & (~exclude_mask) & (df['V'] < 0.5) & (
        (df['wind'] >= 10) | (df['humidity'].fillna(50) < 40)
    )
    other_ortacha = is_other & (~exclude_mask) & (~other_kuchli) & (df['V'] < 1.0) & (
        (df['wind'] >= 8) | (df['humidity'].fillna(50) < 40)
    )
    other_yengil = is_other & (~exclude_mask) & (~other_kuchli) & (~other_ortacha) & (
        (df['V'] < 2.0) & (df['wind'] >= 10) & (df['humidity'].fillna(50) < 40)
    )
    
    # ═══════════════════════════════════════════════════
    # NATIJA
    # ═══════════════════════════════════════════════════
    
    df['dust_severity'] = 'NONE'
    
    # Cho'l
    df.loc[chol_yengil, 'dust_severity'] = 'YENGIL'
    df.loc[chol_ortacha, 'dust_severity'] = 'ORTACHA'
    df.loc[chol_kuchli, 'dust_severity'] = 'KUCHLI'
    
    # Tekislik / tog' oldi
    df.loc[other_yengil, 'dust_severity'] = 'YENGIL'
    df.loc[other_ortacha, 'dust_severity'] = 'ORTACHA'
    df.loc[other_kuchli, 'dust_severity'] = 'KUCHLI'
    
    # Diagnostika ustunlari
    df['is_dust'] = df['dust_severity'].isin(['KUCHLI', 'ORTACHA', 'YENGIL'])
    df['is_fog'] = is_fog
    df['is_snow'] = is_snow
    
    return df

In [ ]:
# Algoritmni qo'llash
df_all = detect_dust_storms_v2(df_all)

# Natijalar
print("📊 YANGILANGAN NATIJALAR (tog'siz + harorat filtri):")
print("=" * 60)

total = len(df_all)
dust = df_all['is_dust'].sum()
fog = df_all['is_fog'].sum()
snow = df_all['is_snow'].sum()

print(f"   Jami kunlar: {total:,}")
print(f"   \n   CHANG BO'RONI: {dust:,} ({dust/total*100:.1f}%)")
for sev in ['KUCHLI', 'ORTACHA', 'YENGIL']:
    count = (df_all['dust_severity'] == sev).sum()
    print(f"     {'🔴' if sev=='KUCHLI' else '🟠' if sev=='ORTACHA' else '🟡'} {sev}: {count:,}")

print(f"\n   CHIQARILGAN:")
print(f"     🌫️ Tuman: {fog:,} ({fog/total*100:.1f}%)")
print(f"     ❄️ Qor bo'roni: {snow:,} ({snow/total*100:.1f}%)")
print(f"     ⚪ Normal: {(total - dust - fog - snow):,}")

In [ ]:
# Terrain type bo'yicha solishtirish
print("\n📊 TERRAIN TYPE BO'YICHA CHANG BO'RONI:")
print("=" * 70)

terrain_stats = df_all.groupby('terrain_type').agg(
    stansiyalar=('station', 'nunique'),
    jami_kunlar=('is_dust', 'count'),
    chang_kunlari=('is_dust', 'sum'),
    kuchli=('dust_severity', lambda x: (x == 'KUCHLI').sum()),
    ortacha=('dust_severity', lambda x: (x == 'ORTACHA').sum()),
    yengil=('dust_severity', lambda x: (x == 'YENGIL').sum()),
    tuman=('is_fog', 'sum'),
    qor=('is_snow', 'sum'),
).round(0)

terrain_stats['chang_%'] = (terrain_stats['chang_kunlari'] / terrain_stats['jami_kunlar'] * 100).round(1)
terrain_stats['tuman_%'] = (terrain_stats['tuman'] / terrain_stats['jami_kunlar'] * 100).round(1)

print(terrain_stats.to_string())

## 5. Vizualizatsiya

In [ ]:
# Terrain type bo'yicha chang bo'roni — bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Chap: absolyut soni
terrain_plot = df_all[df_all['is_dust']].groupby(
    ['terrain_type', 'dust_severity']
).size().unstack(fill_value=0)

colors = {'KUCHLI': '#d32f2f', 'ORTACHA': '#ff9800', 'YENGIL': '#fdd835'}
plot_cols = [c for c in ['KUCHLI', 'ORTACHA', 'YENGIL'] if c in terrain_plot.columns]
terrain_plot[plot_cols].plot(kind='bar', stacked=True, ax=axes[0],
                             color=[colors[c] for c in plot_cols])
axes[0].set_title('Chang bo\'roni hodisalari soni (terrain bo\'yicha)', fontsize=13)
axes[0].set_xlabel('Terrain type')
axes[0].set_ylabel('Hodisalar soni')
axes[0].tick_params(axis='x', rotation=0)

# O'ng: foiz (stansiya boshiga o'rtacha)
terrain_pct = df_all.groupby(['terrain_type', 'station'])['is_dust'].mean().reset_index()
terrain_pct['is_dust'] = terrain_pct['is_dust'] * 100

sns.boxplot(data=terrain_pct, x='terrain_type', y='is_dust', ax=axes[1],
            palette={'chol': '#e65100', 'tekislik': '#1565c0', 'tog_oldi': '#2e7d32'})
axes[1].set_title('Chang bo\'roni kunlari % (stansiya boshiga)', fontsize=13)
axes[1].set_xlabel('Terrain type')
axes[1].set_ylabel('Chang kunlari (%)')

plt.tight_layout()
plt.show()

In [ ]:
# Oylar bo'yicha mavsumiylik — terrain bo'yicha
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

month_names = ['Yan', 'Fev', 'Mar', 'Apr', 'May', 'Iyn',
               'Iyl', 'Avg', 'Sen', 'Okt', 'Noy', 'Dek']

for idx, terrain in enumerate(['chol', 'tekislik', 'tog_oldi']):
    subset = df_all[(df_all['terrain_type'] == terrain) & (df_all['is_dust'])]
    monthly = subset.groupby('Mon').size()
    
    ax = axes[idx]
    bars = ax.bar(range(1, 13), [monthly.get(m, 0) for m in range(1, 13)],
                  color={'chol': '#e65100', 'tekislik': '#1565c0', 'tog_oldi': '#2e7d32'}[terrain],
                  alpha=0.8)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(month_names, rotation=45)
    ax.set_title(f'{terrain.upper()} — mavsumiylik', fontsize=12)
    ax.set_ylabel('Hodisalar soni')

plt.suptitle('Chang bo\'roni mavsumiylik — terrain bo\'yicha', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Top-20 stansiyalar (yangilangan — tog'siz)
station_dust = df_all[df_all['is_dust']].groupby(
    ['station', 'terrain_type', 'dust_severity']
).size().reset_index(name='count')

station_total = station_dust.groupby(['station', 'terrain_type'])['count'].sum().reset_index()
station_total = station_total.sort_values('count', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(14, 8))
terrain_colors = {'chol': '#e65100', 'tekislik': '#1565c0', 'tog_oldi': '#2e7d32'}

bars = ax.barh(range(len(station_total)), station_total['count'],
               color=[terrain_colors.get(t, '#666') for t in station_total['terrain_type']])
ax.set_yticks(range(len(station_total)))
ax.set_yticklabels([f"{row['station']} ({row['terrain_type']})" 
                    for _, row in station_total.iterrows()])
ax.set_xlabel('Hodisalar soni (2011-2020)')
ax.set_title('Top-20 stansiyalar (chang bo\'roni) — rangi = terrain type', fontsize=13)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=t) for t, c in terrain_colors.items()]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

print("\nTop-20 jadval:")
print(station_total.to_string(index=False))

In [ ]:
# Yillar bo'yicha trend
fig, ax = plt.subplots(figsize=(14, 6))

yearly_terrain = df_all[df_all['is_dust']].groupby(['Year', 'terrain_type']).size().unstack(fill_value=0)

yearly_terrain.plot(kind='bar', stacked=True, ax=ax,
                    color=[terrain_colors.get(c, '#666') for c in yearly_terrain.columns])
ax.set_title('Chang bo\'roni hodisalari — yillar bo\'yicha (terrain bo\'yicha)', fontsize=13)
ax.set_xlabel('Yil')
ax.set_ylabel('Hodisalar soni')
ax.legend(title='Terrain')
plt.tight_layout()
plt.show()

## 6. Stansiyalar xaritasi (interaktiv)

In [ ]:
import folium
from folium.plugins import MarkerCluster

# Stansiya boshiga chang kunlari
station_summary = df_all.groupby('station').agg(
    dust_days=('is_dust', 'sum'),
    total_days=('is_dust', 'count'),
    lat=('lat', 'first'),
    lon=('lon', 'first'),
    terrain_type=('terrain_type', 'first'),
    elevation_m=('elevation_m', 'first'),
).reset_index()
station_summary['dust_pct'] = (station_summary['dust_days'] / station_summary['total_days'] * 100).round(1)

# Xarita yaratish
m = folium.Map(location=[41.3, 64.5], zoom_start=6, tiles='CartoDB positron')

# Terrain ranglari
terrain_map_colors = {'chol': 'red', 'tekislik': 'blue', 'tog_oldi': 'green'}

for _, row in station_summary.iterrows():
    if pd.isna(row['lat']) or pd.isna(row['lon']):
        continue
    
    color = terrain_map_colors.get(row['terrain_type'], 'gray')
    
    # Radius = chang kunlari foizi
    radius = max(3, row['dust_pct'] * 1.5)
    
    popup_text = (
        f"<b>{row['station']}</b><br>"
        f"Terrain: {row['terrain_type']}<br>"
        f"Balandlik: {row['elevation_m']:.0f} m<br>"
        f"Chang kunlari: {row['dust_days']:.0f} ({row['dust_pct']}%)<br>"
    )
    
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=radius,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=folium.Popup(popup_text, max_width=200),
        tooltip=f"{row['station']}: {row['dust_pct']}%"
    ).add_to(m)

# Legend
legend_html = '''
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
            background: white; padding: 10px; border-radius: 5px;
            border: 2px solid grey; font-size: 12px;">
    <b>Terrain type:</b><br>
    <span style="color:red;">●</span> Cho'l<br>
    <span style="color:blue;">●</span> Tekislik<br>
    <span style="color:green;">●</span> Tog' oldi<br>
    <br><b>Doira = chang %</b>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

m

## 7. Yakuniy "label" jadvalini saqlash (ML uchun tayyor)

In [ ]:
# ML model uchun label jadvalini saqlash
output_dir = '../analysis/results/'
os.makedirs(output_dir, exist_ok=True)

# 1. Chang bo'roni hodisalari (yangilangan)
dust_events = df_all[df_all['is_dust']][[
    'station', 'date', 'dust_severity', 'terrain_type', 'viloyat',
    'lat', 'lon', 'elevation_m',
    'V', 'VxG', 'Vx8', 'UN', 'U', 'Taav', 'wind', 'humidity'
]].copy()

dust_events.to_csv(f'{output_dir}dust_storm_events_v2.csv', index=False)
print(f"✅ dust_storm_events_v2.csv — {len(dust_events):,} hodisa")

# 2. To'liq ma'lumot (barcha kunlar, label bilan)
df_all.to_csv(f'{output_dir}all_stations_labeled.csv', index=False)
print(f"✅ all_stations_labeled.csv — {len(df_all):,} qator")

# 3. Kunlik umumiy holat (barcha stansiyalar bo'yicha)
daily_summary = df_all.groupby('date').agg(
    stations_with_dust=('is_dust', 'sum'),
    total_stations=('is_dust', 'count'),
    kuchli_count=('dust_severity', lambda x: (x == 'KUCHLI').sum()),
    V_mean=('V', 'mean'),
    wind_mean=('wind', 'mean'),
    humidity_mean=('humidity', 'mean'),
    temp_mean=('Taav', 'mean'),
).round(2)
daily_summary['dust_pct'] = (daily_summary['stations_with_dust'] / daily_summary['total_stations'] * 100).round(1)
daily_summary.to_csv(f'{output_dir}daily_dust_summary.csv')
print(f"✅ daily_dust_summary.csv — {len(daily_summary):,} kun")

print(f"\n💾 Natijalar saqlandi: {output_dir}")

## 8. Xulosa

### 0-bosqichga nisbatan nima o'zgardi:
- ❌ Tog' stansiyalari chiqarildi (13 ta) — qor/tuman, chang emas
- ❄️ Harorat filtri qo'shildi — Taav < 0°C = qor bo'roni
- 🏜️ Cho'l stansiyalari uchun moslantirilgan mezonlar
- 🗺️ Interaktiv xarita yaratildi

### Keyingi qadam (2-bosqich):
- ERA5 reanaliz ma'lumotlarini ulash (gridlangan shamol, namlik)
- Sentinel-5P UVAI bilan validatsiya (2018-2020)
- Feature engineering: lag features, rolling mean, seasonal decomposition
- ML model (XGBoost) qurish

In [ ]:
# Yakuniy statistika
print("\n" + "="*70)
print("  YAKUNIY STATISTIKA")
print("="*70)

print(f"\n  Stansiyalar: {df_all['station'].nunique()} (tog'siz)")
print(f"  Davr: 2011-2020 (10 yil)")
print(f"  Jami kun-stansiya: {len(df_all):,}")
print(f"\n  CHANG BO'RONI:")
print(f"    Jami: {df_all['is_dust'].sum():,} hodisa")
print(f"    Kuchli: {(df_all['dust_severity']=='KUCHLI').sum():,}")
print(f"    O'rtacha: {(df_all['dust_severity']=='ORTACHA').sum():,}")
print(f"    Yengil: {(df_all['dust_severity']=='YENGIL').sum():,}")
print(f"\n  FILTRLANGAN:")
print(f"    Tuman: {df_all['is_fog'].sum():,}")
print(f"    Qor bo'roni: {df_all['is_snow'].sum():,}")
print(f"\n  Cho'l stansiyalari o'rtacha chang %: {terrain_stats.loc['chol', 'chang_%']:.1f}%")
print(f"  Tekislik stansiyalari o'rtacha chang %: {terrain_stats.loc['tekislik', 'chang_%']:.1f}%")